In [ ]:
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')
sys.path.append('/om2/user/bjmedina/auditory-memory/memory/utls/')
from encoders import *
from texture_prior.params import model_params, statistics_set, texture_dataset

resnet_layers = [
        'input_after_preproc', 'conv1_relu1', 'layer1', 'layer2', 'layer3', 'layer4', 'avgpool'
    ]

cnn_layers = [
        'input_after_preproc', 'relu0', 'relu1', 'relu2', 'relu3', 'relu4', 'relufc'
    ]

def make_encoder_name(cfg):
    base = cfg.get("encoder_type", "unknown")

    # include layer if present
    layer = cfg.get("layer", None)
    if layer:
        base += f"-{layer}"

    # include PCA dims if present
    pc = cfg.get("pc_dims", None)
    if pc:
        base += f"-PC{pc}"

    return base

def build_encoder(cfg):

    sys.path.append(f'/om2/user/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')
    print("LOADING FROM", f'/om2/user/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')
       
    etype = cfg["encoder_type"]

    if etype == "kell2018":
        return Kell2018Encoder(
            model_name=cfg["model_name"],
            layer=cfg["layer"],
            sr=cfg["sr"],
            rms_level=cfg["rms_level"],
            duration=cfg["duration"],
            time_avg=cfg["time_avg"],
            device=cfg["device"]
        )
    elif etype == "resnet50":
        return ResNet50Encoder(
            model_name=cfg["model_name"],
            layer=cfg["layer"],
            sr=cfg["sr"],
            rms_level=cfg["rms_level"],
            duration=cfg["duration"],
            time_avg=cfg["time_avg"],
            device=cfg["device"]
        )
    elif etype == 'texture_pca':
        return AudioTextureEncoderPCA(
            statistics_dict=cfg['statistics_dict'],
            pc_dims=cfg['pc_dims'],
            model_params=cfg['model_params'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )
    else:
        raise ValueError(f"Unsupported encoder type: {etype}")

def encode_stimuli(encoder, file_list):
    feats = []
    for filepath in file_list:
        out = encoder(filepath)
        if isinstance(out, dict):
            out = out["embedding"]
        feats.append(out.reshape(-1))
    return torch.stack(feats, dim=0)

In [ ]:
encoder_type = "resnet50" # or "kell2018"
encoder_task =  "word_speaker_audioset" # fixed
pc_dims = None
time_avg = False # or false if you do not want to time average representations

layer = resnet_layers[-2]# which layer do you like?


In [ ]:
encoder_cfg = dict(
    encoder_type=encoder_type,      # e.g. "resnet50"
    model_name=encoder_type,        # same by design
    task=encoder_task,
    statistics_dict=None,
    model_params=model_params,
    pc_dims=pc_dims,
    sr=20000,
    duration=2.0,
    rms_level=0.05,
    time_avg=time_avg,
    device="cuda",
)

# ---- representation-specific fields ----
if encoder_type in ("kell2018", "resnet50"):
    encoder_cfg["layer"] = layer
    assert layer is not None, f"layer required for {encoder_type}"

if encoder_type == "texture":
    encoder_cfg["pc_dims"] = pc_dims
    assert pc_dims is not None, "pc_dims required for texture encoder"

encoder_name = make_encoder_name(encoder_cfg)
print("Encoder name:", encoder_name)

encoder = build_encoder(encoder_cfg)

In [ ]:
#example usage:

sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_p1/*wav")
X = encode_stimuli(encoder, sounds_list[:10])

In [ ]:
X.shape